### 1. Independent Loading and Initial Cleaning Pipelines (Reddit & Text Datasets)

In [20]:
import pandas as pd
from google.colab import drive
import re
import os

drive.mount('/content/drive')

# Define file paths
reddit_file_path = '/content/drive/MyDrive/255 Project Data/df_reddit_cleaned.csv'
text_file_path = '/content/drive/MyDrive/255 Project Data/df_text_cleaned.csv'

# --- Reddit Dataset Pipeline ---
df_reddit = pd.read_csv(reddit_file_path)
# Rename 'target' to 'label'
df_reddit = df_reddit.rename(columns={'target': 'label'})
# Convert integer labels to string for consistency with text labels
df_reddit['label'] = df_reddit['label'].astype(str)

# Map Reddit numerical labels to descriptive names
reddit_label_map = {
    '0': 'Stress',
    '1': 'Depression',
    '2': 'Bipolar',
    '3': 'Personality disorder',
    '4': 'Anxiety'
}
df_reddit['label'] = df_reddit['label'].replace(reddit_label_map)

print(f"Initial df_reddit shape: {df_reddit.shape}")
print(f"Reddit labels: {df_reddit['label'].unique()}")

# Cleaning function for Reddit
def clean_reddit_text(text):
    text = str(text).lower() # Convert to string and lowercase
    return text

df_reddit['cleaned_text'] = df_reddit['text'].apply(clean_reddit_text)


# --- Text Dataset Pipeline ---
df_text = pd.read_csv(text_file_path)
# Rename 'status' to 'label'
df_text = df_text.rename(columns={'status': 'label'})
print(f"Initial df_text shape: {df_text.shape}")
print(f"Text labels: {df_text['label'].unique()}")

# Re-create the matched and OOD subsets for consistency within this notebook
matched_labels_exp5 = ['Anxiety', 'Depression', 'Stress', 'Bipolar', 'Personality disorder']
df_text_matched = df_text[df_text['label'].isin(matched_labels_exp5)].copy()
print(f"df_text_matched shape: {df_text_matched.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Initial df_reddit shape: (5607, 5)
Reddit labels: ['Depression' 'Stress' 'Bipolar' 'Personality disorder' 'Anxiety']
Initial df_text shape: (52681, 3)
Text labels: ['Anxiety' 'Normal' 'Depression' 'Suicidal' 'Stress' 'Bipolar'
 'Personality disorder']
df_text_matched shape: (25686, 3)


### 2. Strip Subreddit-style Leakage Tokens (Reddit Dataset)

In [21]:
# Function to strip subreddit tokens like 'r/depression' or 'u/username'
def strip_subreddit_tokens(text):
    # Remove subreddit mentions (r/subreddit_name)
    text = re.sub(r'r/\S+', '', text, flags=re.IGNORECASE)
    # Remove user mentions (u/username)
    text = re.sub(r'u/\S+', '', text, flags=re.IGNORECASE)
    return text.strip()

print("\n--- Stripping Subreddit Tokens from Reddit Dataset ---")
df_reddit['cleaned_text'] = df_reddit['cleaned_text'].apply(strip_subreddit_tokens)

print("df_reddit after stripping:")
print(df_reddit['cleaned_text'].head())


--- Stripping Subreddit Tokens from Reddit Dataset ---
df_reddit after stripping:
0    welcome to / check-in post - a place to take a...
1    we understand that most people who reply immed...
2    anyone else just miss physical touch? i crave ...
3    i’m just so ashamed. everyone and everything f...
4    i really need a friend. i don't even have a si...
Name: cleaned_text, dtype: object


### 3. Stratified 80/20 Split on Reddit Dataset

In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

print("\n--- Performing Stratified 80/20 Split on Combined Reddit and Text Matched Datasets ---")

# Select relevant columns for combination from both datasets
df_reddit_selected = df_reddit[['cleaned_text', 'label']]

# Cleaning 'statement' and createing 'cleaned_text' for df_text_matched_selected
df_text_matched_processed = df_text_matched.copy()
df_text_matched_processed['cleaned_text'] = df_text_matched_processed['statement'].apply(clean_reddit_text)

# Now select the desired columns from the processed df_text_matched
df_text_matched_selected = df_text_matched_processed[['cleaned_text', 'label']]

# Combine the datasets
combined_data = pd.concat([df_reddit_selected, df_text_matched_selected], ignore_index=True)
print(f"Combined data shape (Reddit + Text Matched): {combined_data.shape}")

# Perform stratified split on the combined data
df_combined_train, df_combined_test = train_test_split(
    combined_data,
    test_size=0.2,
    random_state=42,
    stratify=combined_data['label'] # Stratify using the label column from the combined data
)

# Resetting index might be good practice after splitting
df_combined_train = df_combined_train.reset_index(drop=True)
df_combined_test = df_combined_test.reset_index(drop=True)

print(f"Combined train set shape: {df_combined_train.shape}")
print(f"Combined test set shape: {df_combined_test.shape}")

# Check stratification
print("\nCombined training set label distribution:")
print(df_combined_train['label'].value_counts(normalize=True))
print("\nCombined test set label distribution:")
print(df_combined_test['label'].value_counts(normalize=True))

# Save to parquet files
output_dir = '/content/drive/MyDrive/255 Project Data/'
train_path = os.path.join(output_dir, 'combined_train.parquet')
test_path = os.path.join(output_dir, 'combined_test.parquet')

df_combined_train.to_parquet(train_path, index=False)
df_combined_test.to_parquet(test_path, index=False)
print(f"\nCombined training data saved to: {train_path}")
print(f"Combined testing data saved to: {test_path}")

# Assign to df_reddit_train and df_reddit_test to maintain compatibility
df_reddit_train = df_combined_train
df_reddit_test = df_combined_test


--- Performing Stratified 80/20 Split on Combined Reddit and Text Matched Datasets ---
Combined data shape (Reddit + Text Matched): (31293, 2)
Combined train set shape: (25034, 2)
Combined test set shape: (6259, 2)

Combined training set label distribution:
label
Depression              0.530678
Anxiety                 0.159303
Bipolar                 0.123392
Stress                  0.117800
Personality disorder    0.068826
Name: proportion, dtype: float64

Combined test set label distribution:
label
Depression              0.530596
Anxiety                 0.159291
Bipolar                 0.123502
Stress                  0.117750
Personality disorder    0.068861
Name: proportion, dtype: float64

Combined training data saved to: /content/drive/MyDrive/255 Project Data/combined_train.parquet
Combined testing data saved to: /content/drive/MyDrive/255 Project Data/combined_test.parquet


### 4. Apply SMOTE to the Training Split and Save

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.over_sampling import SMOTE
import pandas as pd
import os
import numpy as np

print("\n--- Applying SMOTE to the Training Split (Reddit Dataset) ---")
X_train_text = df_reddit_train['cleaned_text']
y_train = df_reddit_train['label']

# Vectorize the text data using TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_vectorized = tfidf_vectorizer.fit_transform(X_train_text)
print(f"Original training data shape (vectorized): {X_train_vectorized.shape}")

# Apply SMOTE
smote = SMOTE(random_state=42)
X_resampled_sparse, y_resampled = smote.fit_resample(X_train_vectorized, y_train)
print(f"Resampled training data shape (vectorized): {X_resampled_sparse.shape}")

# Create a DataFrame for the resampled data
X_resampled_dense = X_resampled_sparse.toarray()

# Create column names for the vectorized features for the new dataframe
feature_names = [f'tfidf_feature_{i}' for i in range(X_resampled_dense.shape[1])]

df_resampled = pd.DataFrame(X_resampled_dense, columns=feature_names)
df_resampled['label'] = y_resampled # Add the resampled labels

# Save the resampled DataFrame to parquet
output_dir = '/content/drive/MyDrive/255 Project Data/'
train_smote_path = os.path.join(output_dir, 'train_smote.parquet')
df_resampled.to_parquet(train_smote_path, index=False)
print(f"\nResampled training data (with SMOTE) saved to: {train_smote_path}")
print(f"Shape of train_smote.parquet: {df_resampled.shape}")
print("\nResampled label distribution:")
print(df_resampled['label'].value_counts(normalize=True))



--- Applying SMOTE to the Training Split (Reddit Dataset) ---
Original training data shape (vectorized): (25034, 5000)
Resampled training data shape (vectorized): (66425, 5000)

Resampled training data (with SMOTE) saved to: /content/drive/MyDrive/255 Project Data/train_smote.parquet
Shape of train_smote.parquet: (66425, 5001)

Resampled label distribution:
label
Depression              0.2
Bipolar                 0.2
Stress                  0.2
Personality disorder    0.2
Anxiety                 0.2
Name: proportion, dtype: float64


### 5. Preprocessing Report: Row Counts at Various Steps (Reddit & Text Datasets)

In [26]:
print("\n--- Preprocessing Report ---\n")

# Reddit Dataset
print("Reddit Dataset")
print(f"- Initial rows after loading: {len(df_reddit)}")
print("\nText Dataset")
print(f"- Initial rows after loading: {len(df_text)}")
print(f"- Rows in matched subset (df_text_matched): {len(df_text_matched)}")
print("\nCombined Text Dataset")
print(f"- Rows in combined dataset (df_reddit + df_text_matched): {len(combined_data)}")
print(f"- Rows in combined training set (before SMOTE): {len(df_reddit_train)}")
print(f"- Rows in combined test set: {len(df_reddit_test)}")
print(f"- Rows in combined training set (after SMOTE): {len(df_resampled)}")



--- Preprocessing Report ---

Reddit Dataset
- Initial rows after loading: 5607

Text Dataset
- Initial rows after loading: 52681
- Rows in matched subset (df_text_matched): 25686

Combined Text Dataset
- Rows in combined dataset (df_reddit + df_text_matched): 31293
- Rows in combined training set (before SMOTE): 25034
- Rows in combined test set: 6259
- Rows in combined training set (after SMOTE): 66425
